## 🔥 **QUESTÃO 7.1 — CÓDIGO COMPLETO**

In [9]:
import pandas as pd
import numpy as np

# =========================
# 1. Carregar dados
# =========================
vendas = pd.read_csv('../data/raw/vendas_2023_2024.csv')
produtos = pd.read_csv('../data/processed/produtos_clean.csv')

# =========================
# 2. Padronizar colunas
# =========================
vendas = vendas.rename(columns={'id_product': 'product_id'})
produtos = produtos.rename(columns={'code': 'product_id'})

# Garantir mesmo tipo
vendas['product_id'] = vendas['product_id'].astype(str).str.strip()
produtos['product_id'] = produtos['product_id'].astype(str).str.strip()

# =========================
# 3. Merge
# =========================
df = vendas.merge(produtos, on='product_id', how='left')

# =========================
# 4. Validar merge
# =========================
print("Valores nulos em name:", df['name'].isnull().sum())

# =========================
# 5. Escolher produto válido
# =========================
print("\nProdutos disponíveis:")
print(df['name'].value_counts().head(10))

# 👉 Escolher automaticamente o produto mais frequente (robusto)
produto_escolhido = df['name'].value_counts().idxmax()

print(f"\nProduto escolhido automaticamente: {produto_escolhido}")

df = df[df['name'] == produto_escolhido]

# =========================
# 6. Converter datas
# =========================
df['sale_date'] = pd.to_datetime(
    df['sale_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

df = df.dropna(subset=['sale_date'])

# =========================
# 7. Agregar vendas diárias
# =========================
df_daily = (
    df.groupby(df['sale_date'].dt.date)['qtd']
    .sum()
    .reset_index()
)

df_daily.columns = ['data', 'qtd']
df_daily['data'] = pd.to_datetime(df_daily['data'])

# =========================
# 8. Criar calendário completo
# =========================
datas = pd.date_range(
    start=df_daily['data'].min(),
    end=df_daily['data'].max(),
    freq='D'
)

calendario = pd.DataFrame({'data': datas})

df_daily = calendario.merge(df_daily, on='data', how='left')
df_daily['qtd'] = df_daily['qtd'].fillna(0)

# =========================
# 9. Split treino/teste
# =========================
train = df_daily[df_daily['data'] <= '2023-12-31'].copy()
test = df_daily[df_daily['data'] >= '2024-01-01'].copy()

# =========================
# 10. Baseline (média móvel 7 dias)
# =========================
historico = train['qtd'].tail(7).tolist()

previsoes = []

for i in range(len(test)):
    previsao = np.mean(historico[-7:])
    previsoes.append(previsao)
    historico.append(previsao)

test['previsao'] = previsoes

# =========================
# 11. Avaliação (MAE)
# =========================
mae = np.mean(np.abs(test['qtd'] - test['previsao']))

print(f"\nMAE: {mae:.2f}")

# =========================
# 12. Q7.2 — Soma previsão (01 a 07 Jan)
# =========================
soma_previsao = test[
    (test['data'] >= '2024-01-01') &
    (test['data'] <= '2024-01-07')
]['previsao'].sum()

print(f"\nSoma previsão (01 a 07 Jan): {round(soma_previsao)}")

# =========================
# 13. Visualizar resultados
# =========================
print("\nPrevisões:")
print(test.head(10))

Valores nulos em name: 0

Produtos disponíveis:
name
Cabo de Nylon Delta Velocity Oceanic Abyss     88
Motor Elétrico Torqeedo Velocity Swift 74HP    85
Motor de Popa Honda Torque 228HP               85
Transponder Garmin Drift Vortex Marlin         83
Cabo de Nylon Delta Velocity Core Mako         83
Transponder AIS Vector                         82
Corrente Danforth Zenith Oceanic Torque        82
Transponder Furuno Force Ion                   81
Radar Furuno Swift                             81
Âncora Delta Swift                             80
Name: count, dtype: int64

Produto escolhido automaticamente: Cabo de Nylon Delta Velocity Oceanic Abyss

MAE: 2.59

Soma previsão (01 a 07 Jan): 16

Previsões:
          data  qtd  previsao
363 2024-01-01  0.0  2.142857
364 2024-01-02  0.0  2.448980
365 2024-01-03  0.0  2.798834
366 2024-01-04  0.0  3.198667
367 2024-01-05  0.0  1.655620
368 2024-01-06  0.0  1.892137
369 2024-01-07  0.0  2.019585
370 2024-01-08  0.0  2.308097
371 2024-01-09  

## 🎯 **QUESTÃO 7.2 — VALIDAÇÃO**

### 🔥 Soma da previsão (01 a 07 Jan)

In [ ]:
id="calcq72"
soma_previsao = test[
    (test['data'] >= '2024-01-01') &
    (test['data'] <= '2024-01-07')
]['previsao'].sum()

print(round(soma_previsao))